# 01 · Entrenamiento del primer modelo de detección

**Objetivo:** entrenar un modelo YOLOv8 capaz de detectar jugadores, árbitros y balón en frames de fútbol, partiendo del subconjunto inicial de 152 imágenes etiquetadas manualmente en Roboflow.

**Contexto:** este es el primer modelo del proyecto y servirá como base para el flujo de active learning. Sus resultados no se esperan perfectos, sino suficientes para pre-etiquetar automáticamente el resto del dataset, reduciendo el tiempo de etiquetado total. Se aplica la técnica de fine-tuning sobre el modelo pre-entrenado de Ultralytics en el dataset COCO, lo que permite obtener buenos resultados con un volumen reducido de datos.

**Entrada:** dataset YOLOv8 descargado de Roboflow en `data/dataset/`.

**Salida:** modelo entrenado `best_v1.pt` y métricas de evaluación.

## 1. Configuración del entorno

In [1]:
import torch
from ultralytics import YOLO
from pathlib import Path
import os

REPO_ROOT = Path("../..").resolve()
DATASET_DIR = REPO_ROOT / "data" / "dataset"
DATA_YAML = DATASET_DIR / "data.yaml"

print(f"Directorio del dataset: {DATASET_DIR}")
print(f"Fichero data.yaml: {DATA_YAML}")
print(f"\nGPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

Directorio del dataset: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset
Fichero data.yaml: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset\data.yaml

GPU disponible: True
Dispositivo: NVIDIA GeForce RTX 2060
VRAM total: 6.00 GB


## 2. Verificación y corrección del `data.yaml`

El fichero `data.yaml` define las rutas del dataset y las clases. Roboflow lo genera con rutas relativas que pueden fallar al ejecutar el entrenamiento desde otra ubicación, por lo que se sobrescribe con rutas absolutas para garantizar reproducibilidad.

In [2]:
yaml_content = f"""train: {DATASET_DIR}/train/images
val: {DATASET_DIR}/valid/images
nc: 3
names: ['ball', 'player', 'referee']
"""

with open(DATA_YAML, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print("data.yaml actualizado con rutas absolutas:\n")
print(yaml_content)

data.yaml actualizado con rutas absolutas:

train: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset/train/images
val: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset/valid/images
nc: 3
names: ['ball', 'player', 'referee']



## 3. Verificación del dataset

Antes de entrenar se comprueba que existen las imágenes y etiquetas esperadas.

In [3]:
train_imgs = list((DATASET_DIR / "train" / "images").glob("*.jpg"))
train_lbls = list((DATASET_DIR / "train" / "labels").glob("*.txt"))
valid_imgs = list((DATASET_DIR / "valid" / "images").glob("*.jpg"))
valid_lbls = list((DATASET_DIR / "valid" / "labels").glob("*.txt"))

print(f"Train  · imágenes: {len(train_imgs)} | etiquetas: {len(train_lbls)}")
print(f"Valid  · imágenes: {len(valid_imgs)} | etiquetas: {len(valid_lbls)}")
print(f"Total  · {len(train_imgs) + len(valid_imgs)} imágenes etiquetadas")

Train  · imágenes: 129 | etiquetas: 129
Valid  · imágenes: 23 | etiquetas: 23
Total  · 152 imágenes etiquetadas


## 4. Justificación de hiperparámetros

- **Modelo base:** `yolov8s.pt` (variante small de YOLOv8), buen equilibrio entre precisión y consumo de VRAM, adecuado para los 6 GB de la RTX 2060.
- **Épocas:** 50, suficientes para fine-tuning sin sobreajuste.
- **Tamaño de imagen:** 640×640, valor por defecto que YOLOv8 internamente espera.
- **Batch size:** 16, máximo aprovechable con la VRAM disponible.
- **Workers:** 0, necesario en Windows para evitar problemas de multiprocessing.
- **Patience:** 15, detiene el entrenamiento si no mejora en 15 épocas seguidas.

## 5. Lanzamiento del entrenamiento

El entrenamiento se ejecuta sobre la GPU local. Para 152 imágenes el proceso completo tarda aproximadamente 5 minutos.

In [4]:
model = YOLO("yolov8s.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="runs/detect/models",
    name="detector_v1",
    patience=15,
    save=True,
    plots=True,
    workers=0
)

print(f"\nEntrenamiento completado")
print(f"Resultados guardados en: {results.save_dir}")

New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.53  Python-3.13.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momen

## 6. Análisis de resultados

Tras el entrenamiento, YOLOv8 genera automáticamente varios gráficos y métricas en la carpeta `results.save_dir`. Los más relevantes para la memoria son:

- `results.png`: evolución de pérdida y métricas por época.
- `confusion_matrix.png`: matriz de confusión entre clases.
- `BoxPR_curve.png`: curva precisión-recall.
- `val_batch0_pred.jpg`: ejemplos de predicciones sobre el conjunto de validación.

In [5]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

results_dir = Path(results.save_dir)

graficos = [
    ("results.png", "Evolución del entrenamiento"),
    ("confusion_matrix.png", "Matriz de confusión"),
    ("BoxPR_curve.png", "Curva Precision-Recall"),
    ("val_batch0_pred.jpg", "Predicciones sobre validación")
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, (filename, titulo) in zip(axes.flat, graficos):
    img_path = results_dir / filename
    if img_path.exists():
        img = mpimg.imread(str(img_path))
        ax.imshow(img)
        ax.set_title(titulo, fontsize=12)
    else:
        ax.text(0.5, 0.5, f"No encontrado:\n{filename}", ha="center", va="center")
    ax.axis("off")

plt.tight_layout()
plt.show()

<Figure size 1600x1200 with 4 Axes>

## 7. Métricas finales por clase

Se ejecuta la validación final del modelo sobre el conjunto de validación y se muestran las métricas desglosadas por clase.

In [7]:
metrics = model.val()

print("\nMétricas finales del modelo v1:\n")
print(f"{'Clase':<15} {'Precisión':>10} {'Recall':>10} {'mAP50':>10} {'mAP50-95':>10}")
print("-" * 60)

class_names = model.names
for i, cls in class_names.items():
    p = metrics.box.p[i]
    r = metrics.box.r[i]
    map50 = metrics.box.ap50[i]
    map5095 = metrics.box.ap[i]
    print(f"{cls:<15} {p:>10.3f} {r:>10.3f} {map50:>10.3f} {map5095:>10.3f}")

print("-" * 60)
print(f"{'Promedio':<15} {metrics.box.mp:>10.3f} {metrics.box.mr:>10.3f} {metrics.box.map50:>10.3f} {metrics.box.map:>10.3f}")

Ultralytics 8.4.53  Python-3.13.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 253.332.0 MB/s, size: 47.7 KB)
val: Scanning C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset\valid\labels.cache... 23 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 23/23 4.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.6s/it 5.2s<15.1s
                   all         23        522      0.912      0.818      0.821      0.418
                  ball         22         22      0.909        0.5      0.551      0.111
                player         23        457       0.89      0.976      0.949      0.583
               referee         22         43      0.936      0.977      0.962       0.56
Speed: 2.8ms preprocess, 22.6ms inference, 0.0ms loss, 3.7ms postprocess per image
Results saved to C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\notebooks\fase2_deteccion

## 8. Copia del mejor modelo a la carpeta del proyecto

YOLOv8 guarda el modelo entrenado en `runs/detect/models/detector_v1/weights/best.pt`. Se copia a la carpeta `models/` del proyecto como `best_v1.pt` para conservar también las versiones futuras y facilitar su acceso desde otros notebooks.

In [ ]:
import shutil

source = results_dir / "weights" / "best.pt"
destination = REPO_ROOT / "models" / "best_v1.pt"
destination.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(source, destination)

print(f"Modelo copiado:")
print(f"  Origen:  {source}")
print(f"  Destino: {destination}")
print(f"  Tamaño:  {destination.stat().st_size / (1024*1024):.2f} MB")

## 9. Conclusiones

Tras este notebook:

- Se ha entrenado el primer modelo de detección con 152 imágenes etiquetadas manualmente.
- Los resultados confirman que el fine-tuning sobre YOLOv8s es suficiente para alcanzar buenas métricas en jugadores y árbitros con un dataset reducido.
- La detección del balón presenta un mAP50 inferior, lo que es esperado dado su tamaño reducido y la dificultad inherente del problema. Esto se abordará en iteraciones posteriores mediante refuerzo del dataset y técnicas adicionales.
- El modelo queda disponible en `models/best.pt` y se utilizará en el siguiente notebook para el auto-etiquetado de los frames restantes.